# Projeto Final — Análise Exploratória de Dados (Olist)
## Grupo: Gustavo Pires Bogéa / Haroé Silva de Jesus / Hélio Serrano Brandão Junior

## 1. Apresentação da Base e Perguntas de Negócio

### 1.1. Origem dos Dados

Os dados foram extraídos do dataset público do **Olist** no Kaggle, cobrindo o e-commerce brasileiro. A combinação dessas tabelas permite mapear desde a realização da compra até a entrega do pedido e a avaliação final feita pelo cliente.

Bases originais utilizadas
a. Base de Clientes: 'df_customers'
b. Base de Perdas: 'df_orders'
c. Base de Itens de Pedidos: 'df_itens'
d. Base de Pagamentos: 'df_payments'
e. Base de Reavaliação: 'df_reviews'
f. Base de Produtos: 'df_products'
g. Base de Categorias de produto: 'df_categories'

Bases derivadas
a. Base de Pedidos de Clientes: 'df_orders_customers' - junção das bases originais 'df_orders' com 'df_customers' relacionando clientes e pedidos.
b. Base de Pedidos de clientes e Itens: 'df_entrega' - junção da base derivada 'df_orders_customers' e com a base original 'df_itens, incluindo a relação por itens. 
c. Base de Itens e Produtos: 'df_itens_produtos' - junção das bases originais 'df_itens' e 'df_products'

#### 1.1.1 Base de Clientes ('df_customers')
Armazena informações cadastrais e geográficas dos compradores.
* **`customer_id`**: Chave gerada a cada pedido (ligação à tabela de pedidos).
* **`customer_unique_id`**: Identificador único do cliente (utilizado para analisar compras recorrentes).
* **`customer_zip_code_prefix`**: Código postal do cliente.
* **`customer_city`**: Cidade de residência do cliente.
* **`customer_state`**: Sigla do Estado (UF) do cliente.

#### 1.1.2 Base de Pedidos ('df_orders')
Centraliza o ciclo de vida e a esteira de status de cada transação.
* **`order_id`**: Identificador único do pedido.
* **`customer_id`**: Chave de ligação com a tabela de clientes.
* **`order_status`**: Status atual do pedido (`delivered`, `canceled`, etc.).
* **`order_purchase_timestamp`**: Data e hora da realização da compra.
* **`order_delivered_customer_date`**: Data e hora da entrega efetiva ao cliente.
* **`order_estimated_delivery_date`**: Data estimada de entrega.

#### 1.1.3 Base de Itens do Pedido ('df_items')
Detalha os produtos e valores associados a cada item contido num pedido.
* **`order_id`**: Identificador do pedido.
* **`order_item_id`**: Sequencial do item dentro do pedido (ex: 1, 2, 3).
* **`product_id`**: Identificador único do produto vendido.
* **`seller_id`**: Identificador único do vendedor.
* **`price`**: Preço unitário do produto (em R$).
* **`freight_value`**: Valor do frete associado ao item (em R$).

#### 1.1.4 Base de Pagamentos ('df_payments')
Registra os métodos financeiros e condições de parcelamento.
* **`order_id`**: Identificador do pedido.
* **`payment_type`**: Método de pagamento utilizado (`credit_card`, `boleto`, `voucher`, `debit_card`).
* **`payment_installments`**: Número de parcelas escolhidas.
* **`payment_value`**: Valor total pago na transação.

#### 1.1.5 Base de Avaliações ('df_reviews')
Contém os feedbacks diretos deixados pelos clientes após a entrega.
* **`review_id`**: Identificador único da avaliação.
* **`order_id`**: Identificador do pedido avaliado.
* **`review_score`**: Nota atribuída pelo cliente (1 a 5).
* **`review_comment_message`**: Comentário/feedback em texto do cliente.

#### 1.1.6 Base de Produtos ('df_products')
Contém as características físicas e a categoria de cada produto listado na plataforma.
* **`product_id`**: Identificador único do produto.
* **`product_category_name`**: Nome da categoria do produto em português.
* **`product_weight_g`**: Peso do produto em gramas.
* **`product_length_cm` / `product_height_cm` / `product_width_cm`**: Dimensões físicas do produto.

#### 1.1.7 Base de Categorias ('df_categories')
Tabela auxiliar de mapeamento e tradução das categorias de produtos.
* **`product_category_name`**: Nome original da categoria em português.
* **`product_category_name_english`**: Nome traduzido da categoria para inglês.

### 1.2. Importação Bibliotecas e Configurações do Notebook

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# Configuração de estilo visual
sns.set_theme(style="whitegrid")
plt.rcParams['font.size'] = 10

### 1.3. Obtenção dos dados

In [ ]:
# Leitura das 7 bases de dados
df_customers = pd.read_csv('../dados/olist_customers_dataset.csv')
df_orders = pd.read_csv('../dados/olist_orders_dataset.csv')
df_items = pd.read_csv('../dados/olist_order_items_dataset.csv')
df_payments = pd.read_csv('../dados/olist_order_payments_dataset.csv')
df_reviews = pd.read_csv('../dados/olist_order_reviews_dataset.csv')
df_products = pd.read_csv('../dados/olist_products_dataset.csv')
df_categories = pd.read_csv('../dados/product_category_name_translation.csv')

print("Todas as 7 bases foram carregadas com sucesso!")

### 1.4. PERGUNTAS DO NEGÓCIO
1. **Prazos e Frete:** Qual a relação entre o custo do frete e o tempo real de entrega entre os diferentes estados?
2. **Meios de Pagamento e Ticket:** Como o valor médio do pedido varia em função do tipo de pagamento e do parcelamento?
3. **Outliers de Frete:** Qual o comportamento do status dos pedidos e tempo de entrega ao longo do tempo?
4. **Recorrência:** Clientes recorrentes possuem comportamento de compra diferente dos clientes pontuais?

### 1.5. Estrutura e Granularidade das Bases
Antes de realizar os cruzamentos entre as tabelas, vamos verificar a estrutura e a granularidade de cada base. Isso é importante porque uma mesma informação pode aparecer várias vezes em algumas tabelas. Por exemplo, um pedido pode possuir vários itens e também vários pagamentos. Portanto, em vez de criar um único DataFrame com todas as tabelas, construiremos DataFrames específicos de acordo com cada pergunta de negócio, evitando a duplicação de registros e a distorção das métricas.

In [ ]:
# Tradução das categorias dos produtos #A TABELA DE PRODUTOS ESTÁ SENDO UTILIZADA NA BASE?
df_products = df_products.merge(
    df_categories,
    on='product_category_name',
    how='left'
)

print("Categorias dos produtos traduzidas.")

#### 1.5.1. Verificação da estrutura das bases

Primeiro, vamos verificar a quantidade de linhas e colunas de cada tabela. Essa informação ajuda a identificar o tamanho das bases e, principalmente, perceber que elas possuem diferentes níveis de granularidade.

In [ ]:
print("orders:", df_orders.shape)
print("customers:", df_customers.shape)
print("items:", df_items.shape)
print("payments:", df_payments.shape)
print("products:", df_products.shape)
print("reviews:", df_reviews.shape)
print("categories:", df_categories.shape)

#### 1.5.2. Verificação da granularidade

Agora vamos verificar se order_id é único em cada tabela. Em orders, esperamos um registro por pedido. Já em items e payments, um mesmo pedido pode aparecer várias vezes, pois um pedido pode conter vários itens e pode possuir mais de um pagamento.

In [ ]:
print("order_id único em orders:",
      df_orders['order_id'].is_unique)

print("order_id único em items:",
      df_items['order_id'].is_unique)

print("order_id único em payments:",
      df_payments['order_id'].is_unique)

Interpretação: A tabela orders possui um registro único para cada pedido. Já items e payments possuem múltiplos registros para um mesmo order_id. Por isso, juntar essas tabelas diretamente poderia multiplicar registros quando um pedido possui vários itens e vários pagamentos. Para evitar esse problema, as tabelas serão combinadas de acordo com a necessidade de cada análise.

#### 1.5.3 Construção dos DataFrames para análise

Como as tabelas possuem diferentes níveis de granularidade, não utilizaremos um único DataFrame com todas as informações.

Os cruzamentos serão realizados de acordo com cada pergunta de negócio, preservando a granularidade adequada para cada análise.

##### 1.5.3.1. Pedidos e clientes

Primeiro, vamos relacionar os pedidos aos seus respectivos clientes.

A tabela `orders` possui um registro por pedido e `customers` possui as informações do cliente associado ao pedido. Portanto, esperamos uma relação de um para um entre essas duas tabelas por `customer_id`.

In [ ]:
df_orders_customers = df_orders.merge(
    df_customers,
    on='customer_id',
    how='left',
    validate='one_to_one'
)

print("Orders:", df_orders.shape)
print("Orders + Customers:", df_orders_customers.shape)

**Interpretação:** O primeiro cruzamento entre `orders` e `customers` manteve as 99.441 linhas da tabela de pedidos. Isso confirma que o relacionamento utilizado preservou a granularidade de um registro por pedido.

##### 1.5.3.2. Pedidos, clientes e itens

Agora adicionaremos os itens dos pedidos. Diferentemente da relação anterior, um pedido pode possuir vários itens. Portanto, esperamos uma relação de um para muitos (`one-to-many`).

Após esse cruzamento, a granularidade do DataFrame passa de **um registro por pedido** para **um registro por item do pedido**.

In [ ]:
df_entrega = df_orders_customers.merge(
    df_items,
    on='order_id',
    how='left',
    validate='one_to_many'
)

print("Orders + Customers:", df_orders_customers.shape)
print("Orders + Customers + Items:", df_entrega.shape)

**Interpretação:** Após adicionar a tabela `items`, o número de registros aumentou de 99.441 para 113.425. Isso ocorre porque um mesmo pedido pode conter vários itens. Portanto, o DataFrame `df_entrega` passa a ter como granularidade um registro por item do pedido, e não mais um registro por pedido.

In [ ]:
df_entrega[['order_id', 'product_id', 'price', 'freight_value']].head(10)

### Verificação de um pedido com múltiplos itens

Para visualizar na prática a mudança de granularidade, vamos identificar um pedido que possui vários itens e observar como ele aparece no DataFrame após o cruzamento.

In [ ]:
pedido_exemplo = (
    df_entrega.groupby('order_id')
    .size()
    .sort_values(ascending=False)
    .index[0]
)

print("Pedido escolhido:", pedido_exemplo)

df_entrega[
    df_entrega['order_id'] == pedido_exemplo
][['order_id', 'product_id', 'price', 'freight_value']]

##### 1.5.3.3. Itens e produtos

Para complementar as informações dos itens, vamos relacioná-los aos dados dos produtos por meio de `product_id`.

A tabela `order_items` pode possuir vários registros para o mesmo produto, pois um produto pode ser vendido em diferentes pedidos. Já a tabela `products` possui um registro por produto.

Portanto, esperamos uma relação de muitos para um (`many-to-one`).

In [ ]:
df_itens_produtos = df_items.merge(
    df_products,
    on='product_id',
    how='left',
    validate='many_to_one'
)

print("Items:", df_items.shape)
print("Items + Products:", df_itens_produtos.shape)

O cruzamento manteve a quantidade de registros porque cada item está associado a um único produto. A relação `many-to-one` foi validada com sucesso.

## 2. Diagnóstico da Qualidade dos Dados

Antes de realizar as análises, vamos verificar a qualidade e a estrutura dos dados utilizados.

Como cada pergunta de negócio utiliza DataFrames com diferentes granularidades, o diagnóstico será realizado considerando a base correspondente a cada análise.

Nesta primeira etapa, vamos analisar o `df_entrega`, utilizado nas análises relacionadas a prazo, frete e estado.

### 2.1. Dimensões, tipos e uso de memória

In [ ]:
df_entrega.info()

**Interpretação:** O DataFrame `df_entrega` possui 113.425 registros e 18 colunas, com aproximadamente 15,6 MB de uso de memória. A maior parte das colunas está no formato `object`, incluindo as datas, que posteriormente poderão ser convertidas para o formato `datetime` para permitir cálculos relacionados a prazos de entrega.

Também foram identificados valores ausentes em algumas colunas, principalmente nas informações relacionadas à aprovação e às datas de entrega. Esses valores serão investigados na próxima etapa do diagnóstico antes de definir qualquer estratégia de tratamento.

### 2.2. Valores ausentes

In [ ]:
missing = pd.DataFrame({
    'quantidade': df_entrega.isna().sum(),
    'percentual': (df_entrega.isna().mean() * 100).round(2)
})

missing = missing[missing['quantidade'] > 0].sort_values(
    'percentual',
    ascending=False
)

missing

#### 2.2.1. Investigação dos valores ausentes em order_items

As colunas relacionadas aos itens apresentam exatamente a mesma quantidade de valores ausentes. Vamos verificar se esses registros correspondem a pedidos que não possuem informações na tabela `order_items`.

Essa verificação é importante para diferenciar valores ausentes originados nos dados de origem daqueles que surgiram como consequência do cruzamento entre as tabelas.

In [ ]:
colunas_items = [
    'order_item_id',
    'product_id',
    'seller_id',
    'shipping_limit_date',
    'price',
    'freight_value'
]

df_entrega[df_entrega['order_item_id'].isna()][colunas_items].head()

#### 2.2.2. Investigação dos valores ausentes nas datas do pedido

As datas relacionadas ao processo de entrega apresentam valores ausentes em diferentes proporções. Antes de definir qualquer tratamento, vamos verificar a situação dos pedidos e identificar se os valores ausentes estão relacionados ao status do pedido.

In [ ]:
df_entrega['order_status'].value_counts(dropna=False)

#### 2.2.3. Relação entre status do pedido e datas ausentes

Como `df_entrega` possui granularidade de item, a análise das datas e do status será realizada utilizando `df_orders_customers`, que possui uma linha por pedido.

Dessa forma, cada pedido será contabilizado uma única vez, evitando que pedidos com vários itens tenham peso maior no diagnóstico.

In [ ]:
status_datas = df_orders_customers.groupby('order_status').agg(
    total_pedidos=('order_id', 'count'),
    sem_data_transportadora=('order_delivered_carrier_date', lambda x: x.isna().sum()),
    sem_data_entrega=('order_delivered_customer_date', lambda x: x.isna().sum())
)

status_datas

#### 2.2.4. Investigação dos pedidos entregues sem data de entrega

Embora a maioria dos pedidos com status `delivered` possua as datas de entrega preenchidas, foram identificados 8 pedidos sem `order_delivered_customer_date`.

Vamos verificar esses registros antes de definir o tratamento, pois eles não poderão ser utilizados no cálculo do prazo real de entrega enquanto a data estiver ausente.

In [ ]:
df_orders_customers[
    (df_orders_customers['order_status'] == 'delivered') &
    (df_orders_customers['order_delivered_customer_date'].isna())
][[
    'order_id',
    'order_status',
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]]

**Interpretação:** Foram identificados 8 pedidos com status `delivered` que não possuem a data efetiva de entrega ao cliente. Como essa informação não pode ser determinada de forma confiável a partir dos dados disponíveis, esses valores não serão preenchidos artificialmente.

Para a análise de prazo real de entrega, serão considerados posteriormente apenas os pedidos que possuam as datas necessárias para o cálculo. Dessa forma, preservamos os dados originais e evitamos introduzir informações estimadas sem justificativa.

### 2.3. Registros duplicados

Como `df_orders_customers` possui uma linha por pedido, vamos verificar se existem registros completamente duplicados nessa base.

Essa verificação é importante para identificar possíveis repetições dos mesmos registros antes das etapas de transformação e análise.

In [ ]:
duplicados_orders = df_orders_customers.duplicated().sum()

print("Registros duplicados:", duplicados_orders)

**Interpretação:** Não foram identificados registros completamente duplicados em `df_orders_customers`. Portanto, não será necessário realizar uma remoção de duplicidades nessa base.

### 2.4. Verificação de categorias inconsistentes

Vamos verificar as categorias existentes em `order_status` para identificar possíveis diferenças de grafia, espaços ou outras inconsistências que possam representar a mesma categoria.

In [ ]:
df_orders_customers['order_status'].value_counts(dropna=False)

**Interpretação:** As categorias de `order_status` apresentam nomenclatura padronizada, sem diferenças aparentes de grafia, capitalização ou espaços que indiquem categorias duplicadas. Portanto, não é necessário realizar tratamento de padronização nessa variável.

#### 2.4.1. Verificação das categorias de estado

Como o estado do cliente será utilizado nas análises de frete e prazo, vamos verificar as categorias presentes em `customer_state` e identificar possíveis valores inconsistentes.

In [ ]:
df_orders_customers['customer_state'].value_counts(dropna=False)

#### 2.4.2. Verificação de espaços e caps

Além da inspeção das categorias, vamos verificar se existem valores que seriam alterados ao remover espaços ou padronizar as letras para maiúsculas.

In [ ]:
estados_inconsistentes = df_orders_customers[
    df_orders_customers['customer_state'] !=
    df_orders_customers['customer_state'].str.strip().str.upper()
][['customer_state']]

estados_inconsistentes.drop_duplicates()

### 2.5. Verificação de valores inválidos

Nesta etapa, vamos verificar se existem valores numéricos que não são compatíveis com o significado das variáveis utilizadas na análise.

Como `price` representa o preço do item e `freight_value` representa o valor do frete, valores negativos seriam inconsistentes com essas definições.

In [ ]:
valores_invalidos = {
    'price_negativo': (df_entrega['price'] < 0).sum(),
    'freight_negativo': (df_entrega['freight_value'] < 0).sum()
}

pd.Series(valores_invalidos)

**Interpretação:** Não foram identificados valores negativos nas variáveis `price` e `freight_value`. Portanto, não foi identificado, nesta verificação, nenhum valor incompatível com o significado dessas variáveis.

### 2.6. Identificação de Outliers

In [ ]:
# Configuração de estilo visual
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 10)

# =========================================================
# 2.2.1 VERIFICAÇÃO DE NORMALIDADE (HISTOGRAMAS E ASSIMETRIA)
# =========================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Distribuição do Preço dos Produtos (df_items)
sns.histplot(df_items['price'], bins=50, kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title(f"Distribuição de Preço (Assimetria: {df_items['price'].skew():.2f})")
axes[0, 0].set_xlabel("Preço (R$)")

# 2. Distribuição do Valor do Frete (df_items)
sns.histplot(df_items['freight_value'], bins=50, kde=True, ax=axes[0, 1], color='salmon')
axes[0, 1].set_title(f"Distribuição de Frete (Assimetria: {df_items['freight_value'].skew():.2f})")
axes[0, 1].set_xlabel("Valor do Frete (R$)")

# 3. Distribuição do Valor do Pagamento (df_payments)
sns.histplot(df_payments['payment_value'], bins=50, kde=True, ax=axes[1, 0], color='lightgreen')
axes[1, 0].set_title(f"Distribuição de Pagamento (Assimetria: {df_payments['payment_value'].skew():.2f})")
axes[1, 0].set_xlabel("Valor Pago (R$)")

# 4. Distribuição do Peso do Produto (df_products)
sns.histplot(df_products['product_weight_g'].dropna(), bins=50, kde=True, ax=axes[1, 1], color='orchid')
axes[1, 1].set_title(f"Distribuição do Peso (Assimetria: {df_products['product_weight_g'].skew():.2f})")
axes[1, 1].set_xlabel("Peso (gramas)")

plt.tight_layout()
plt.show()

# =========================================================
# 2.2.2 APLICAÇÃO DO MÉTODO IQR (ADEQUADO PARA DISTRIBUIÇÕES ASSIMÉTRICAS)
# =========================================================
print("=== APLICAÇÃO DO MÉTODO IQR PARA VARIÁVEIS ASSIMÉTRICAS ===")

# A) Outliers de Frete por Grupo de Comparação (Estado do Cliente)
df_items_cust = df_items.merge(df_orders[['order_id', 'customer_id']], on='order_id')
df_items_cust = df_items_cust.merge(df_customers[['customer_id', 'customer_state']], on='customer_id')

def detecta_outliers_iqr_grupo(grupo, coluna):
    q1 = grupo[coluna].quantile(0.25)
    q3 = grupo[coluna].quantile(0.75)
    iqr = q3 - q1
    limite_superior = q3 + 1.5 * iqr
    outliers = grupo[grupo[coluna] > limite_superior]
    
    return pd.Series({
        'Total_Registros': len(grupo),
        'Q1': round(q1, 2),
        'Q3': round(q3, 2),
        'Limite_Sup_IQR': round(limite_superior, 2),
        'Qtd_Outliers': len(outliers),
        '%_Outliers': round((len(outliers) / len(grupo)) * 100, 2)
    })

print("\n--- Outliers de Frete Agrupados por Estado (Top 10 UFs com mais anomalias) ---")
resumo_frete_estado = df_items_cust.groupby('customer_state').apply(detecta_outliers_iqr_grupo, coluna='freight_value').sort_values(by='Qtd_Outliers', ascending=False)
print(resumo_frete_estado.head(10))

# B) Outliers Globais via IQR para Preço e Valor de Pagamento
def resumo_iqr_global(df, coluna, nome_base):
    q1 = df[coluna].quantile(0.25)
    q3 = df[coluna].quantile(0.75)
    iqr = q3 - q1
    lim_sup = q3 + 1.5 * iqr
    outliers = df[df[coluna] > lim_sup]
    print(f"\n--- Outliers Globais via IQR: Base {nome_base} ({coluna}) ---")
    print(f"Q1: R$ {q1:.2f} | Q3: R$ {q3:.2f} | Limite Superior IQR: R$ {lim_sup:.2f}")
    print(f"Registros acima do limite: {len(outliers):,} ({len(outliers)/len(df)*100:.2f}%)")

resumo_iqr_global(df_items, 'price', 'df_items')
resumo_iqr_global(df_payments, 'payment_value', 'df_payments')

### 2.6.1 Justificativa Metodológica e Relatório de Outliers

#### Teste de Normalidade e Escolha do Método
* **Diagnóstico de Distribuição:** A análise visual dos histogramas aliada ao cálculo do coeficiente de assimetria (*skewness*) confirma que as variáveis numéricas das bases (`price`, `freight_value`, `payment_value` e `product_weight_g`) **não seguem uma distribuição normal** (apresentam assimetria positiva acentuada com cauda longa à direita).
* **Definição da Métrica:** Em distribuições não normais/assimétricas, a média e o desvio padrão são severamente distorcidos por valores extremos, inviabilizando o uso do Z-Score. Desta forma, o método do **Intervalo Interquartil (IQR)** foi selecionado como a abordagem estatística adequada, definindo os limites como:
  $$\text{Limite Superior} = Q_3 + 1.5 \times \text{IQR}$$

#### Contextualização por Grupo de Comparação (Frete por Estado)
* A avaliação do valor do frete foi executada agrupando as transações pelo Estado de destino do cliente (`customer_state`).
* **Justificativa:** Custos logísticos no Brasil variam drasticamente conforme a distância geográfica dos centros de distribuição (localizados majoritariamente no Sudeste). Definir um único limite de frete global classificaria erroneamente fretes normais da Região Norte como *outliers*. O agrupamento por UF garante um diagnóstico justo dentro da realidade de cada mercado regional.

#### Interpretação Negocial das Anomalias
* Os *outliers* identificados em preço e valor pago não representam erros de digitação na base de dados, mas sim transações reais de produtos de elevado valor unitário ou compras em lote.

In [ ]:
# ==============================================================================
# Plot dos histogramas com estimativa de densidade de kernel (KDE)
# Automatizado para todos os DataFrames do projeto
# ==============================================================================

# Dicionário com os DataFrames mapeados pelo grupo
dataframes_projeto = {
    'df_orders_customers': df_orders_customers,
    'df_items': df_items,
    'df_payments': df_payments,
    'df_entrega': df_entrega
}

for nome_df, df_atual in dataframes_projeto.items():
    # Seleção de colunas numéricas excluindo IDs e códigos sequenciais
    colunas_numericas = df_atual.select_dtypes(include=["number"]).columns
    colunas_numericas = [c for c in colunas_numericas if not c.endswith('_id') and c != 'order_item_id']
    
    if len(colunas_numericas) == 0:
        continue

    print(f"\n📊 Gerando histogramas para: {nome_df}")

    num_cols = len(colunas_numericas)
    cols_per_row = 3
    rows = (num_cols + cols_per_row - 1) // cols_per_row

    plt.figure(figsize=(16, 4 * rows))

    for i, col in enumerate(colunas_numericas, 1):
        plt.subplot(rows, cols_per_row, i)
        
        # Histograma com estimativa de densidade de kernel (KDE)
        sns.histplot(df_atual[col].dropna(), kde=True, bins=30, color='skyblue')
        
        # Linhas de referência para Média e Mediana
        media_val = df_atual[col].mean()
        mediana_val = df_atual[col].median()
        
        plt.axvline(media_val, color='red', linestyle='--', alpha=0.8, label=f'Média ({media_val:.1f})')
        plt.axvline(mediana_val, color='darkorange', linestyle='-', alpha=0.8, label=f'Mediana ({mediana_val:.1f})')
        
        plt.title(f'Distribuição de {col} ({nome_df})', fontsize=11)
        plt.xlabel(col)
        plt.ylabel('Frequência')
        plt.legend(fontsize=9)
        plt.grid(axis='y', linestyle='--', alpha=0.4)

    plt.tight_layout()
    plt.show()

## 3. Limpeza e Transformação

### 3.1 CONVERSÃO DE TIPOS DE DADOS

In [ ]:
colunas_datas = [
    'order_purchase_timestamp', 
    'order_approved_at', 
    'order_delivered_carrier_date', 
    'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]

for col in colunas_datas:
    df_orders[col] = pd.to_datetime(df_orders[col])

print("Datas convertidas com sucesso!")

### 3.2 CRIAÇÃO DE DATAFRAMES CONSOLIDADOS (MERGES)

In [ ]:
# Base 1: df_entrega (Foco Logístico)
# Unificação: Orders + Customers + Items
df_entrega = df_orders.merge(df_customers, on='customer_id', how='inner')
df_entrega = df_entrega.merge(df_items, on='order_id', how='inner')

# Base 2: df_vendas_completo (Foco Financeiro e Satisfação)
# Unificação: df_entrega + Payments + Reviews + Products + Categories
df_vendas_completo = df_entrega.merge(df_payments, on='order_id', how='left')
df_vendas_completo = df_vendas_completo.merge(df_reviews[['order_id', 'review_score']], on='order_id', how='left')
df_vendas_completo = df_vendas_completo.merge(df_products, on='product_id', how='left')
df_vendas_completo = df_vendas_completo.merge(df_categories, on='product_category_name', how='left')

print(f"Base 'df_entrega' criada: {df_entrega.shape[0]:,} linhas")
print(f"Base 'df_vendas_completo' criada: {df_vendas_completo.shape[0]:,} linhas")

### 3.3 CRIAÇÃO DAS COLUNAS DERIVADAS (MÍNIMO 3 EXIGIDAS)

In [ ]:
# Coluna Derivada 1: Tempo Real de Entrega em Dias
df_entrega['prazo_entrega_dias'] = (
    df_entrega['order_delivered_customer_date'] - df_entrega['order_purchase_timestamp']
).dt.total_seconds() / (24 * 3600)

# Coluna Derivada 2: Diferença em Relação à Estimativa (Dias de Atraso ou Antecipação)
# Valores positivos = Entrega Adiantada | Valores negativos = Entrega Com Atraso
df_entrega['diferenca_estimativa_dias'] = (
    df_entrega['order_estimated_delivery_date'] - df_entrega['order_delivered_customer_date']
).dt.total_seconds() / (24 * 3600)

# Coluna Derivada 3: Razão entre Frete e Preço (Peso do Frete no Produto)
df_entrega['razao_frete_preco'] = df_entrega['freight_value'] / df_entrega['price']

# Coluna Derivada 4 (Uso do np.where): Status de Pontualidade
df_entrega['status_pontualidade'] = np.where(
    df_entrega['diferenca_estimativa_dias'] >= 0, 'No Prazo', 'Com Atraso'
)

# Coluna Derivada 5 (Uso de pd.qcut): Categorização do Frete em Quartis
df_entrega['categoria_custo_frete'] = pd.qcut(
    df_entrega['freight_value'], 
    q=4, 
    labels=['Frete Barato', 'Frete Médio-Baixo', 'Frete Médio-Alto', 'Frete Caro']
)

print("Colunas derivadas criadas:")
print(df_entrega[['prazo_entrega_dias', 'diferenca_estimativa_dias', 'razao_frete_preco', 'status_pontualidade', 'categoria_custo_frete']].head())

### 3.4 Justificativa das Decisões de Limpeza e Transformação

1. Tratamento de Tipos e Registros Ausentes
* **Conversão de Datas:** Todas as variáveis temporais do ecossistema de pedidos foram convertidas de `object` para `datetime64`. Isso viabilizou operações matemáticas vetorizadas para medir o ciclo de vida do pedido.
* **Manutenção dos Valores Nulos de Entrega:** Não removemos nem imputamos valores médios nas datas de entrega ausentes (`order_delivered_customer_date`). A ausência desses dados reflete o estado real do negócio (pedidos cancelados, em processamento ou ainda em trânsito). Substituí-los por médias distorceria a métrica de eficiência logística.

2. Estratégia de Agrupamento das Bases
* A criação de **`df_entrega`** consolida o nível de granularidade do item e do cliente, permitindo responder às perguntas sobre tempo de transporte e frete por Estado.
* A base **`df_vendas_completo`** integra os meios de pagamento, notas de avaliação e dados das categorias de produtos, centralizando a visão multifacetada da experiência do cliente.

3. Recursos Criados (Engenharia de Variáveis)
* **`prazo_entrega_dias`:** Calcula o tempo corrido entre a compra e o recebimento pelo cliente.
* **`diferenca_estimativa_dias`:** Quantifica a margem de erro da estimativa lograda pela plataforma.
* **`razao_frete_preco`:** Mensura o impacto do frete sobre o valor nominal do produto.
* **`status_pontualidade` (via `np.where`):** Classifica cada entrega como "No Prazo" ou "Com Atraso".
* **`categoria_custo_frete` (via `pd.qcut`):** Discretiza o valor do frete em 4 faixas estatísticas baseadas em quantis.

## 4. Análise Exploratória

### 4.1. PERGUNTA 1 - Relação entre frete e tempo real de entrega

#### 4.1.1. Análise dos pedidos e cálculo da correlação entre custo e prazo

In [ ]:
# Agregação dos itens para obter o frete total por pedido
frete_pedido = (
    df_items
    .groupby('order_id')
    .agg(
        frete_total=('freight_value', 'sum'),
        valor_produtos=('price', 'sum'),
        quantidade_itens=('order_item_id', 'count')
    )
    .reset_index()
)

# Merge entre pedidos/clientes e o frete agregado por pedido
pedidos_q1 = (
    df_orders
    .merge(
        df_customers[['customer_id', 'customer_state']],
        on='customer_id',
        how='left',
        validate='one_to_one'
    )
    .merge(
        frete_pedido,
        on='order_id',
        how='inner',
        validate='one_to_one'
    )
)

# Considerar somente pedidos entregues com datas válidas
pedidos_q1 = pedidos_q1[
    (pedidos_q1['order_status'] == 'delivered') &
    (pedidos_q1['order_purchase_timestamp'].notna()) &
    (pedidos_q1['order_delivered_customer_date'].notna())
].copy()

# Tempo real entre a compra e a entrega
pedidos_q1['prazo_real_dias'] = (
    pedidos_q1['order_delivered_customer_date']
    - pedidos_q1['order_purchase_timestamp']
).dt.total_seconds() / (24 * 3600)

# Remoção de valores inconsistentes
pedidos_q1 = pedidos_q1[
    (pedidos_q1['prazo_real_dias'] >= 0) &
    (pedidos_q1['frete_total'] >= 0)
].copy()

# Região geográfica
mapeamento_regiao = {
    'SP': 'Sudeste', 'RJ': 'Sudeste', 'MG': 'Sudeste', 'ES': 'Sudeste',
    'PR': 'Sul', 'SC': 'Sul', 'RS': 'Sul',
    'BA': 'Nordeste', 'PE': 'Nordeste', 'CE': 'Nordeste',
    'MA': 'Nordeste', 'PB': 'Nordeste', 'PI': 'Nordeste',
    'AL': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
    'MT': 'Centro-Oeste', 'MS': 'Centro-Oeste',
    'GO': 'Centro-Oeste', 'DF': 'Centro-Oeste',
    'AM': 'Norte', 'PA': 'Norte', 'RO': 'Norte',
    'TO': 'Norte', 'AC': 'Norte', 'AP': 'Norte', 'RR': 'Norte'
}

pedidos_q1['regiao'] = pedidos_q1['customer_state'].map(mapeamento_regiao)

# Normalização por Z-score utilizando NumPy
pedidos_q1['frete_zscore'] = (
    pedidos_q1['frete_total'] - np.mean(pedidos_q1['frete_total'])
) / np.std(pedidos_q1['frete_total'])

pedidos_q1['prazo_zscore'] = (
    pedidos_q1['prazo_real_dias'] - np.mean(pedidos_q1['prazo_real_dias'])
) / np.std(pedidos_q1['prazo_real_dias'])

# Correlação geral entre frete e prazo
correlacao_geral = np.corrcoef(
    pedidos_q1['frete_total'],
    pedidos_q1['prazo_real_dias']
)[0, 1]

print(f'Pedidos analisados: {len(pedidos_q1)}')
print(f'Correlação entre frete e prazo real: {correlacao_geral:.3f}')

#### 4.1.2. Agregação por estado

In [ ]:
resumo_estado_q1 = (
    pedidos_q1
    .groupby(['customer_state', 'regiao'])
    .agg(
        total_pedidos=('order_id', 'nunique'),
        frete_medio=('frete_total', 'mean'),
        frete_mediano=('frete_total', 'median'),
        prazo_medio=('prazo_real_dias', 'mean'),
        prazo_mediano=('prazo_real_dias', 'median'),
        prazo_minimo=('prazo_real_dias', 'min'),
        prazo_maximo=('prazo_real_dias', 'max')
    )
    .reset_index()
    .sort_values('prazo_medio', ascending=False)
)

resumo_estado_q1.head(10)

#### 4.1.3. Relação Frete total x Prazo de entrega

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=pedidos_q1.sample(
        min(15000, len(pedidos_q1)),
        random_state=42
    ),
    x='frete_total',
    y='prazo_real_dias',
    hue='regiao',
    alpha=0.45
)

plt.title('Relação entre frete total e prazo real de entrega')
plt.xlabel('Frete total do pedido (R$)')
plt.ylabel('Prazo real de entrega (dias)')
plt.legend(title='Região')
plt.tight_layout()
plt.show()

**Leitura do gráfico:** A dispersão mostra se pedidos com fretes mais altos também apresentam prazos maiores. A grande dispersão indica que o frete, isoladamente, não explica todo o prazo de entrega.

#### 4.1.4. Comparação entre estados

In [ ]:
top_estados = (
    pedidos_q1['customer_state']
    .value_counts()
    .head(15)
    .index
)

resumo_top_estados = resumo_estado_q1[
    resumo_estado_q1['customer_state'].isin(top_estados)
].sort_values('prazo_medio', ascending=False)

fig, ax1 = plt.subplots(figsize=(12, 6))

sns.barplot(
    data=resumo_top_estados,
    x='customer_state',
    y='prazo_medio',
    color='steelblue',
    ax=ax1
)

ax1.set_title('Prazo médio e frete médio por estado')
ax1.set_xlabel('Estado')
ax1.set_ylabel('Prazo médio (dias)')

ax2 = ax1.twinx()
ax2.plot(
    resumo_top_estados['customer_state'],
    resumo_top_estados['frete_medio'],
    color='crimson',
    marker='o',
    linewidth=2
)

ax2.set_ylabel('Frete médio (R$)', color='crimson')
plt.tight_layout()
plt.show()

**Leitura do gráfico:** Os estados localizados em regiões mais distantes dos principais centros de distribuição tendem a apresentar prazos médios maiores. O frete também tende a ser mais elevado nesses estados, embora existam diferenças entre eles.

#### 4.1.5. Pivot table e heatmap

In [ ]:
pedidos_q1['faixa_frete'] = pd.qcut(
    pedidos_q1['frete_total'],
    q=4,
    labels=[
        'Frete baixo',
        'Frete médio-baixo',
        'Frete médio-alto',
        'Frete alto'
    ],
    duplicates='drop'
)

pivot_frete_prazo = pd.pivot_table(
    pedidos_q1,
    values='prazo_real_dias',
    index='customer_state',
    columns='faixa_frete',
    aggfunc='mean'
)

plt.figure(figsize=(12, 10))

sns.heatmap(
    pivot_frete_prazo,
    annot=True,
    fmt='.1f',
    cmap='YlOrRd',
    cbar_kws={'label': 'Prazo médio (dias)'}
)

plt.title('Prazo médio por estado e faixa de frete')
plt.xlabel('Faixa de frete')
plt.ylabel('Estado')
plt.tight_layout()
plt.show()

**Leitura do gráfico:** A tabela dinâmica permite observar se, dentro de cada estado, pedidos com frete mais alto possuem prazos diferentes. As cores mais intensas representam maiores prazos médios de entrega.

#### 4.1.6. Comparação regional

In [ ]:
plt.figure(figsize=(10, 6))

sns.boxplot(
    data=pedidos_q1,
    x='regiao',
    y='prazo_real_dias',
    order=[
        'Sudeste',
        'Sul',
        'Centro-Oeste',
        'Nordeste',
        'Norte'
    ]
)

plt.title('Distribuição do prazo real de entrega por região')
plt.xlabel('Região')
plt.ylabel('Prazo real de entrega (dias)')
plt.ylim(0, pedidos_q1['prazo_real_dias'].quantile(0.99))
plt.tight_layout()
plt.show()

**Leitura do gráfico:** As regiões Norte e Nordeste tendem a apresentar maior variabilidade e prazos mais elevados. O Sudeste apresenta, em geral, entregas mais rápidas e concentradas, provavelmente pela maior proximidade dos centros de distribuição.

#### Resposta da pergunta de negócio 1

Existe uma relação positiva entre o custo do frete e o tempo real de entrega quando analisamos os pedidos individualmente. Essa associação é confirmada pela correlação de 0,167, indicando que pedidos com fretes mais altos tendem, em média, a apresentar prazos de entrega maiores. Entretanto, essa relação é fraca e não é perfeita, pois o prazo final também depende diretamente da distância, da região de destino, do vendedor e da eficiência da operação logística envolvida. 

A análise por estado evidencia fortes diferenças regionais. Estados mais afastados dos principais centros de distribuição tendem a sofrer com fretes mais caros e prazos médios elevados. As regiões Norte e Nordeste, por exemplo, geralmente apresentam maior tempo de trânsito e alta variabilidade na entrega, enquanto as regiões Sudeste e Sul tendem a apresentar o melhor desempenho logístico do país. 

Portanto, o custo do frete funciona como um indicador parcial da complexidade logística de um pedido, mas não possui força estatística para ser utilizado isoladamente na previsão exata do prazo de entrega.

### 4.2. PERGUNTA 2 - Como o valor médio do pedido varia em função do tipo de pagamento e do parcelamento?

#### 4.2.1. Preparação e consolidação dos dados

In [ ]:
# Valor total do pedido no nível do pedido
valor_pedido_q2 = (
    df_items
    .groupby('order_id')
    .agg(
        valor_produtos=('price', 'sum'),
        frete_total=('freight_value', 'sum'),
        quantidade_itens=('order_item_id', 'count')
    )
    .reset_index()
)

valor_pedido_q2['valor_pedido'] = (
    valor_pedido_q2['valor_produtos'] +
    valor_pedido_q2['frete_total']
)

# Consolidação dos pagamentos no nível do pedido.
# Quando há mais de um pagamento, os tipos são combinados.
pagamentos_pedido_q2 = (
    df_payments
    .groupby('order_id')
    .agg(
        pagamento_total=('payment_value', 'sum'),
        tipo_pagamento=('payment_type', lambda x: ' + '.join(sorted(x.dropna().unique()))),
        parcelas=('payment_installments', 'max'),
        quantidade_pagamentos=('payment_type', 'count')
    )
    .reset_index()
)

# Merge entre o valor dos itens e as informações de pagamento
analise_q2 = (
    valor_pedido_q2
    .merge(
        pagamentos_pedido_q2,
        on='order_id',
        how='inner',
        validate='one_to_one'
    )
)

# Remoção de registros inválidos
analise_q2 = analise_q2[
    (analise_q2['valor_pedido'] >= 0) &
    (analise_q2['pagamento_total'] >= 0) &
    (analise_q2['parcelas'] > 0)
].copy()

# Normalização do valor do pedido utilizando NumPy
media_valor = np.mean(analise_q2['valor_pedido'])
desvio_valor = np.std(analise_q2['valor_pedido'])

analise_q2['valor_pedido_zscore'] = (
    analise_q2['valor_pedido'] - media_valor
) / desvio_valor

print(f"Pedidos analisados: {len(analise_q2)}")
print(f"Valor médio do pedido: R$ {analise_q2['valor_pedido'].mean():.2f}")
print(f"Valor mediano do pedido: R$ {analise_q2['valor_pedido'].median():.2f}")

#### 4.2.2. Evidência numérica por tipo de pagamento e parcelamento

In [ ]:
resumo_pagamento_q2 = (
    analise_q2
    .groupby('tipo_pagamento')
    .agg(
        total_pedidos=('order_id', 'nunique'),
        valor_medio=('valor_pedido', 'mean'),
        valor_mediano=('valor_pedido', 'median'),
        valor_minimo=('valor_pedido', 'min'),
        valor_maximo=('valor_pedido', 'max'),
        parcelas_medias=('parcelas', 'mean')
    )
    .reset_index()
    .sort_values('valor_medio', ascending=False)
)

resumo_parcelamento_q2 = (
    analise_q2
    .groupby('parcelas')
    .agg(
        total_pedidos=('order_id', 'nunique'),
        valor_medio=('valor_pedido', 'mean'),
        valor_mediano=('valor_pedido', 'median'),
        tipo_pagamento_mais_frequente=(
            'tipo_pagamento',
            lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
        )
    )
    .reset_index()
    .sort_values('parcelas')
)

display(resumo_pagamento_q2)
display(resumo_parcelamento_q2.head(15))

#### 4.2.3. Valor médio por tipo de pagamento

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=resumo_pagamento_q2,
    x="tipo_pagamento",
    y="valor_medio",
    hue="tipo_pagamento",
    palette="viridis",
    legend=False,
)

plt.title("Valor médio do pedido por tipo de pagamento")
plt.xlabel("Tipo de pagamento")
plt.ylabel("Valor médio do pedido (R$)")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

**Leitura do gráfico:** O valor médio varia entre os tipos de pagamento. Essa diferença pode indicar perfis de consumo distintos, mas não permite afirmar que o método de pagamento cause pedidos maiores ou menores.

#### 4.2.4. Valor médio por número de parcelas

In [ ]:
parcelas_grafico = resumo_parcelamento_q2[
    resumo_parcelamento_q2['total_pedidos'] >= 100
].copy()

plt.figure(figsize=(12, 6))

sns.lineplot(
    data=parcelas_grafico,
    x='parcelas',
    y='valor_medio',
    marker='o',
    color='darkblue'
)

plt.title('Valor médio do pedido conforme o número de parcelas')
plt.xlabel('Número de parcelas')
plt.ylabel('Valor médio do pedido (R$)')
plt.xticks(parcelas_grafico['parcelas'])
plt.tight_layout()
plt.show()

**Leitura do gráfico:** O gráfico mostra como o ticket médio se comporta conforme aumenta o número de parcelas. Foram exibidas apenas faixas com pelo menos 100 pedidos, reduzindo a influência de grupos muito pequenos.

#### 4.2.5. Pivot table e heatmap

In [ ]:
analise_q2['faixa_parcelamento'] = pd.cut(
    analise_q2['parcelas'],
    bins=[0, 1, 3, 6, 10, 24],
    labels=[
        '1 parcela',
        '2 a 3 parcelas',
        '4 a 6 parcelas',
        '7 a 10 parcelas',
        '11 a 24 parcelas'
    ]
)

pivot_pagamento_parcelas = pd.pivot_table(
    analise_q2,
    values='valor_pedido',
    index='tipo_pagamento',
    columns='faixa_parcelamento',
    aggfunc='mean',
    observed=False
)

plt.figure(figsize=(12, 6))

sns.heatmap(
    pivot_pagamento_parcelas,
    annot=True,
    fmt='.2f',
    cmap='YlGnBu',
    cbar_kws={'label': 'Valor médio do pedido (R$)'}
)

plt.title('Valor médio do pedido por tipo de pagamento e faixa de parcelamento')
plt.xlabel('Faixa de parcelamento')
plt.ylabel('Tipo de pagamento')
plt.tight_layout()
plt.show()

**Leitura do gráfico:** O heatmap permite analisar simultaneamente o efeito do tipo de pagamento e da faixa de parcelamento. As células mais escuras representam combinações associadas a maiores valores médios de pedido.

#### 4.2.6. Distribuição de valores por tipo de pagamento

In [ ]:
tipos_com_volume = (
    analise_q2['tipo_pagamento']
    .value_counts()
    .loc[lambda x: x >= 100]
    .index
)

dados_boxplot_q2 = analise_q2[
    analise_q2['tipo_pagamento'].isin(tipos_com_volume)
].copy()

limite_superior = dados_boxplot_q2['valor_pedido'].quantile(0.99)

plt.figure(figsize=(11, 6))

sns.boxplot(
    data=dados_boxplot_q2,
    x="tipo_pagamento",
    y="valor_pedido",
    hue="tipo_pagamento",
    showfliers=False,
    palette="Set2",
    legend=False,
)

plt.ylim(0, limite_superior)
plt.title('Distribuição do valor dos pedidos por tipo de pagamento')
plt.xlabel('Tipo de pagamento')
plt.ylabel('Valor do pedido (R$)')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

**Leitura do gráfico:** O boxplot mostra a mediana, a dispersão e a concentração dos valores dos pedidos em cada modalidade. A visualização foi limitada ao percentil 99 para facilitar a comparação sem que valores extremos ocultem as diferenças entre os grupos.

#### Resposta da pergunta de negócio 2

O valor médio do pedido varia de acordo com o tipo de pagamento e com o número de parcelas. A tabela `resumo_pagamento_q2` permite identificar qual modalidade apresenta o maior e o menor ticket médio, enquanto `resumo_parcelamento_q2` mostra a variação do valor conforme o parcelamento.

A análise deve considerar também a mediana e a quantidade de pedidos de cada grupo, pois poucos pedidos de alto valor podem elevar significativamente a média. Em geral, pedidos parcelados no cartão de crédito tendem a apresentar maior valor, já que o parcelamento facilita a aquisição de produtos mais caros. Entretanto, essa associação é descritiva e não permite concluir que o parcelamento seja a causa do aumento do valor do pedido.

A consolidação foi realizada no nível do pedido para evitar a duplicação causada por pedidos com vários itens ou múltiplos registros de pagamento. O uso do `groupby` com agregação múltipla, do `merge` e da `pivot_table` permitiu comparar simultaneamente o tipo de pagamento, o parcelamento e o valor médio dos pedidos.

### 4.3. PERGUNTA 3 - Qual o comportamento do status dos pedidos e tempo de entrega ao longo do tempo?

Objetivo: Mapear a evolução mensal dos prazos de entrega, atrasos e status de envio

In [ ]:
# 0. Atribuição do DataFrame consolidado de entregas
df = df_entrega.copy()

# 1. Mapeamento explícito da coluna 'regiao' (resolve o KeyError)
mapeamento_regiao = {
    'SP': 'Sudeste', 'RJ': 'Sudeste', 'MG': 'Sudeste', 'ES': 'Sudeste',
    'PR': 'Sul', 'SC': 'Sul', 'RS': 'Sul',
    'BA': 'Nordeste', 'PE': 'Nordeste', 'CE': 'Nordeste', 'MA': 'Nordeste', 
    'PB': 'Nordeste', 'PI': 'Nordeste', 'AL': 'Nordeste', 'RN': 'Nordeste', 'SE': 'Nordeste',
    'MT': 'Centro-Oeste', 'MS': 'Centro-Oeste', 'GO': 'Centro-Oeste', 'DF': 'Centro-Oeste',
    'AM': 'Norte', 'PA': 'Norte', 'RO': 'Norte', 'TO': 'Norte', 'AC': 'Norte', 'AP': 'Norte', 'RR': 'Norte'
}
df['regiao'] = df['customer_state'].map(mapeamento_regiao)

# 2. Criação de métricas temporais e colunas derivadas
df['ano_mes'] = df['order_purchase_timestamp'].dt.to_period('M')
df['ano_mes_str'] = df['ano_mes'].astype(str)

# Filtro para período com volume estável de vendas (2017 a agosto de 2018)
df_p3 = df[(df['ano_mes_str'] >= '2017-01') & (df['ano_mes_str'] <= '2018-08')].copy()

# 3. Agregação Pandas Múltipla (.agg) por Mês/Ano
df_resumo_temporal = df_p3.groupby('ano_mes_str').agg(
    total_pedidos=('order_id', 'nunique'),
    prazo_medio_real=('prazo_entrega_dias', 'mean'),
    prazo_mediano_real=('prazo_entrega_dias', 'median'),
    pct_atrasados=('status_pontualidade', lambda x: round((x == 'Com Atraso').mean() * 100, 2)),
    pct_cancelados=('order_status', lambda x: round((x == 'canceled').mean() * 100, 2))
).reset_index()

# 4. Pivot Table: Evolução da Taxa de Atraso (%) por Região
pivot_atraso_regiao = pd.pivot_table(
    df_p3,
    values='status_pontualidade',
    index='ano_mes_str',
    columns='regiao',
    aggfunc=lambda x: round((x == 'Com Atraso').mean() * 100, 2)
)

print("--- Resumo do Desempenho Logístico Mensal (Primeiros Meses) ---")
print(df_resumo_temporal.head(5))

print("\n--- Pivot Table: Taxa de Atraso (%) por Região ---")
print(pivot_atraso_regiao.head(5))

# ============================================================================================
# VISUALIZAÇÃO GRÁFICA DA PERGUNTA 3
# ============================================================================================

# FIGURA 1: Evolução do Volume de Pedidos vs. Taxa de Entregas com Atraso
fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

sns.barplot(data=df_resumo_temporal, x='ano_mes_str', y='total_pedidos', ax=ax1, color='lightskyblue', alpha=0.6)
sns.lineplot(data=df_resumo_temporal, x='ano_mes_str', y='pct_atrasados', ax=ax2, color='crimson', marker='o', linewidth=2.5)

ax1.set_title('Evolução Mensal do Volume de Pedidos vs. Taxa de Entregas com Atraso (%)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Mês/Ano da Compra')
ax1.set_ylabel('Total de Pedidos Realizados', color='navy')
ax2.set_ylabel('Taxa de Pedidos com Atraso (%)', color='crimson')
ax1.tick_params(axis='x', rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# FIGURA 2: Matriz Temporal da Taxa de Atraso (%) por Região Geográfica
plt.figure(figsize=(12, 5))
sns.heatmap(pivot_atraso_regiao.T, cmap='YlOrRd', annot=True, fmt=".1f", cbar_kws={'label': '% de Pedidos Atrasados'})
plt.title('Taxa de Atraso na Entrega (%) por Região Geográfica ao Longo do Tempo', fontsize=12, fontweight='bold')
plt.xlabel('Mês/Ano da Compra')
plt.ylabel('Região Geográfica')
plt.tight_layout()
plt.show()

#### Resposta e Análise Negocial da Pergunta 3

##### Como se comporta o status dos pedidos e o tempo de entrega ao longo do tempo?

1. **Tendência Geral do Tempo de Entrega:**
   * Durante o primeiro semestre de 2017, a plataforma apresentou uma tendência contínua de ganho de eficiência operacional: o tempo médio de entrega caiu de **~18 dias em janeiro/2017 para ~11 dias no meio do ano**.

2. **Gargalo Logístico na Sazonalidade (Efeito Black Friday e Festas):**
   * O gráfico de barras e linha evidencia o principal ponto de inflexão operacional da plataforma: o pico de vendas ocorrido em **novembro de 2017 (Black Friday)** sobrecarregou a capacidade de despacho e transporte.
   * Como resultado, a **taxa de atrasos na entrega saltou de 4% para picos acima de 15% entre dezembro de 2017 e março de 2018**, demonstrando vulnerabilidade no dimensionamento de frota e parceiros logísticos para momentos de alta demanda.

3. **Disparidade Regional Temporal:**
   * A análise por *heatmap* confirma que o impacto dos atrasos nos períodos de pico foi desproporcional entre as regiões: enquanto o **Sudeste manteve taxas de atraso abaixo de 8%**, as regiões **Norte e Nordeste atingiram picos superiores a 20% a 25% de atraso** nas entregas no início de 2018.

### 4.4. PERGUNTA 4 - Clientes recorrentes possuem comportamento de compra diferente dos clientes pontuais?

Para responder a essa pergunta, vamos identificar a quantidade de pedidos
realizados por cada cliente utilizando o `customer_unique_id`.

Como o `df_orders_customers` possui um registro por pedido, podemos agrupar
os pedidos por cliente e identificar quais clientes realizaram apenas uma
compra e quais realizaram duas ou mais compras.

In [ ]:
pedidos_cliente = (
    df_orders_customers
    .groupby('customer_unique_id')
    .size()
    .reset_index(name='quantidade_pedidos')
)

pedidos_cliente.head()

O agrupamento acima transforma a informação que estava no nível de pedido em uma visão no nível do cliente. Assim, cada linha passa a representar um `customer_unique_id` e a quantidade de pedidos realizados por ele.

A partir da quantidade de pedidos, vamos classificar os clientes em dois grupos: clientes pontuais, que realizaram apenas uma compra, e clientes recorrentes, que realizaram duas ou mais compras.

In [ ]:
pedidos_cliente['tipo_cliente'] = np.where(
    pedidos_cliente['quantidade_pedidos'] > 1,
    'Recorrente',
    'Pontual'
)

Essa classificação permitirá comparar posteriormente o comportamento de compra dos dois grupos.

In [ ]:
pedidos_cliente['tipo_cliente'].value_counts()

A distribuição acima mostra a quantidade de clientes classificados como pontuais e recorrentes. Essa informação permite entender o tamanho de cada grupo antes de realizar as comparações de comportamento.

O agrupamento acima permite identificar a quantidade de pedidos realizados por cada cliente. A partir dessa informação, podemos separar os clientes em dois grupos: aqueles que realizaram apenas uma compra e aqueles que realizaram mais de uma compra.

A classificação resultou em 93.099 clientes pontuais e 2.997 clientes recorrentes. Essa distribuição mostra que os clientes pontuais representam a maior parte da base analisada, enquanto uma parcela menor realizou compras recorrentes.

In [ ]:
valor_pedido = (
    df_items
    .groupby('order_id')['price']
    .sum()
    .reset_index(name='valor_pedido')
)

valor_pedido.head()

Como a tabela `df_items` possui um registro para cada item, um mesmo pedido pode aparecer várias vezes. Para calcular o valor total de cada pedido, vamos somar o preço dos seus itens agrupando pelo `order_id`.

Agora vamos relacionar o valor de cada pedido ao respectivo cliente e, consequentemente, ao grupo de recorrência ao qual esse cliente pertence.

Dessa forma, teremos uma base que permite comparar o valor dos pedidos realizados por clientes pontuais e recorrentes.

In [ ]:
analise_recorrencia = (
    df_orders_customers[
        ['order_id', 'customer_unique_id']
    ]
    .merge(valor_pedido, on='order_id', how='inner')
    .merge(
        pedidos_cliente[
            ['customer_unique_id', 'tipo_cliente']
        ],
        on='customer_unique_id',
        how='left'
    )
)

analise_recorrencia.head()

In [ ]:
ticket_medio = (
    analise_recorrencia
    .groupby('tipo_cliente')['valor_pedido']
    .mean()
    .round(2)
)

ticket_medio

Para facilitar a comparação entre os dois grupos de clientes, vamos representar graficamente o ticket médio dos pedidos de clientes pontuais e recorrentes.

In [ ]:
ticket_medio.plot(
    kind='bar',
    title='Ticket médio por tipo de cliente',
    xlabel='Tipo de cliente',
    ylabel='Valor médio do pedido (R$)'
)

plt.xticks(rotation=0)
plt.show()

O gráfico mostra que o ticket médio dos pedidos realizados por clientes pontuais foi de R$ 138,62, enquanto o ticket médio dos clientes recorrentes foi de R$ 124,91.

Assim, na base analisada, os clientes pontuais apresentaram um ticket médio aproximadamente R$ 13,71 superior ao dos clientes recorrentes. Essa diferença descreve o comportamento observado na base, mas não permite afirmar que a recorrência seja a causa da diferença no valor dos pedidos.

## 5. Conclusões, Limitações e Recomendações Estratégicas

### 5.1 Principais Achados de Negócio

1. **Relação entre Frete, Tempo de Entrega e Disparidade Regional (Pergunta 1):**
   A análise geográfica revelou uma forte assimetria logística e financeira entre as regiões do Brasil. Estados do Sul e Sudeste (como SP, RJ, MG e PR) concentram a maioria das vendas e apresentam os menores custos médios de frete combinados aos menores tempos reais de entrega. Em contrapartida, estados das regiões Norte e Nordeste enfrentam valores de frete significativamente mais elevados e prazos médios de entrega substancialmente mais longos. Essa correlação positiva entre custo de frete e tempo de transporte reflete a dependência de malhas rodoviárias/aéreas de longa distância que partem dos polos vendedores concentrados no Centro-Sul.

2. **Comportamento dos Meios de Pagamento, Parcelamento e Ticket Médio (Pergunta 2):**
   O cartão de crédito consolida-se como o meio de pagamento preferencial e de maior ticket médio na plataforma, sendo o principal viabilizador de compras de maior valor por meio do parcelamento. Observou-se uma tendência clara de aumento no valor médio do pedido à medida que o número de parcelas aumenta, evidenciando o papel do crédito na expansão do poder de compra dos consumidores. Em contraste, pagamentos à vista — como boleto bancário e cartão de débito — apresentam tickets médios mais baixos e representam escolhas orientadas a compras pontuais ou de menor desembolso imediato. O uso de *vouchers* (cupons/estornos) aparece majoritariamente associado a pagamentos complementares.

3. **Eficiência Logística Gradual (Pergunta 3):**
   Ao longo do primeiro semestre de 2017, a plataforma demonstrou evolução na curva de aprendizado operacional, reduzindo o tempo médio de entrega de **~18 dias (janeiro/2017) para ~11 dias (meados de 2017)**.

4. **Gargalo de Sazonalidade — Efeito Black Friday (Pergunta 3):**
   O salto expressivo no volume de vendas registrado em **novembro de 2017 (Black Friday)** pode ter sobrecarregado a capacidade de transporte e expedição das parceiras logísticas. Como consequência direta, a taxa de entregas com atraso saltou de **~4% para picos superiores a 15% entre dezembro de 2017 e março de 2018**.

5. **Vulnerabilidade Regional nos Períodos de Pico (Pergunta 3):**
   A análise temporal por região revelou que a sobrecarga operacional afetou desproporcionalmente o **Norte e Nordeste**, onde as taxas de atraso ultrapassaram **20% a 25%** nos meses pós-Black Friday, enquanto o Sudeste manteve a taxa de atrasos abaixo de 8% mesmo durante os picos.

6. **Comportamento de Clientes Recorrentes (Pergunta 4):**
   A análise mostrou que a maior parte dos clientes realizou apenas uma compra, com 93.099 clientes pontuais e 2.997 recorrentes. Na comparação do valor médio dos produtos por pedido, os clientes pontuais apresentaram R$ 138,62, enquanto os clientes recorrentes apresentaram R$ 124,91. Essa diferença indica um comportamento distinto entre os grupos na base analisada, embora não permita estabelecer uma relação de causa entre recorrência e valor do pedido.

De forma geral, a análise permitiu identificar padrões relevantes no comportamento dos pedidos, especialmente em relação ao desempenho logístico ao longo do tempo, ao impacto das modalidades financeiras no ticket médio e às diferenças observadas entre clientes pontuais e recorrentes. Os resultados devem ser interpretados como evidências descritivas da base analisada, servindo como apoio para a identificação de oportunidades de melhoria e para a definição de análises futuras mais aprofundadas.


### 5.2 Limitações da Análise

* **Dados de Origem/Mapeamento Financeiro e Frete:** A base não especifica se o valor do frete praticado possui subsídio (frete grátis parcial ou total promovido pela plataforma/vendedor) ou se reflete integralmente a tabela da transportadora, limitando análises de elasticidade-preço do frete.
* **Granularidade do Parcelamento sem Juros:** Não há indicação direta se o parcelamento via cartão de crédito em muitas parcelas acarretou taxa de juros repassada ao consumidor final ou se foi absorvida pelo seller/marketplace.
* **Falta de Dados de Rotas e Armazenagem:** A base não especifica o modal de transporte (aéreo ou rodoviário) nem a localização do centro de distribuição (*seller*) de onde o item partiu, impossibilitando mapear o gargalo por transportadora específica.
* **Ausência de Indicadores do Vendedor (SLA de Despacho):** Não é possível separar com precisão quanto do atraso total ocorreu no tempo de separação do produto pelo vendedor versus o tempo em trânsito pela transportadora.
* **Janela Temporal Restrita:** A base abrange dados até agosto de 2018, impedindo a análise do comportamento completo do quarto trimestre de 2018 para validar se as ações corretivas pós-Black Friday de 2017 foram efetivas.


### 5.3 Recomendações Estratégicas e Próximos Passos

1. **Políticas de Subsídio de Frete e Tabela Regionalizada:**
   Incentivar políticas de frete grátis ou frete fixo condicionadas a um valor mínimo de compra (*threshold* de pedido), mitigando o impacto do alto custo de frete para consumidores das regiões Norte e Nordeste sem comprometer a margem das operações.

2. **Otimização dos Meios de Pagamento e Oferta de Crédito:**
   Promover condições facilitadas de parcelamento para categorias de produtos com ticket elevado, visando incentivar a elevação do valor médio do pedido. Adicionalmente, implementar opções modernas de pagamento à vista com menor custo de intercâmbio (como o Pix, em bases atuais) para capturar o público hoje usuário de boleto.

3. **Planejamento de Capacidade para Eventos Sazonais:**
   Estabelecer acordos de nível de serviço (SLA) com frotas de contingência e contratação temporária de capacidade de transporte com antecedência mínima de 90 dias para a *Black Friday*.

4. **Estratégia de *Fulfillment* Regionalizada:**
   Criar *hubs* ou centros de distribuição avançados nas regiões Norte e Nordeste para descentralizar o estoque, reduzindo a dependência da malha de longa distância vinda do Sudeste nos períodos de pico.

5. **Ajuste Dinâmico na Promessa de Entrega:**
   Utilizar a taxa de atraso histórica por região para calibrar automaticamente a estimativa de entrega exibida ao cliente (*checkout*) durante os meses de novembro a fevereiro, alinhando as expectativas e preservando a experiência do consumidor.